# **Driven Pendulum with Wall Impact**
## FEM Implementation using NGSolve
---------------------------------------

The following implementation is a simple example of an elastic pendulum with wall impact using the Finite Element Method (FEM) with NGSolve.
Following references were used to implement the model:
- [Elastic Pendulum - NGS Tutorial 2024](https://docu.ngsolve.org/ngs24/tutorials/00_dynamics.html)
- [Contact Problems - NGS Docu Interactive Tutorial](https://docu.ngsolve.org/latest/i-tutorials/unit-6.2-contact/contact.html)
- [An Interactive Introduction to the Finite Element Method, Joachim Schöberl, TU Wien, ASC](https://jschoeberl.github.io/iFEM/intro.html)
- [Nonlinear Elasticity - NGS Docu](https://docu.ngsolve.org/ngs24/SaS/nonlinearelasticity.html)

----------------------------------------------

In [ ]:
from netgen.occ import *
from ngsolve import *

---------------------------------------

### **1.Physical Problem and Governing PDE**

#### **1.1 Physical Model**

- The model implements a 2D deformable pendulum that consist of a rod with a circular mass at the end.
- The pendulum rotates around a fixed pivot point that is the center of the rod's top edge (`rotation` edge)
- The gravitational force acts in negative y-direction.
- The pendulum can collide with a vertical rigid wall at an angular position $q=0$.
- The pendulum is driven by an external torque that is applied via normal traction on the rod's top edge (`rotation` edge).
- Due to large deformations, a nonlinear material law (hyperelastic / Neo-Hookean) is used to model the elasticity.

|Pendulum Geometry|Boundary and Initial Condition|
|-----------------|------------------------------|
|![](images/img_dimensions.png)|![](images/img_setup.png)|

#### **1.2 Governing PDE: Nonlinear Elastodynamics**



In the reference configuration $\Omega_0 \subset \mathbb{R}^2$:

$$\rho_0 \ddot{\mathbf{u}} = \nabla_0 \cdot \mathbf{P} + \mathbf{b} \quad \text{in } \Omega_0$$

 - $\mathbf{u(\mathbf{X},t)}$: Displacement field
 - $\rho_0$: Density in reference configuration
 - $\mathbf{P}$: First Piola-Kirchhoff stress tensor $\mathbf{P} = \frac{\partial \Psi}{\partial \mathbf{F}}$
    - $\Psi$: Strain energy density function (Neo-Hookean material model)
    - $\mathbf{F}$: Deformation gradient $\mathbf{F} = \mathbf{I} + \nabla_0 \mathbf{u}$
 - $\mathbf{b}$: Body force per unit reference volume (e.g. gravity)

#### **1.3 Constitutive Model: Neo-Hookean Hyperelastic Material**

For large deformations, the linear elastic model is not appropriate. Instead, we use the Neo-Hookean hyperelastic material model.

Let

$$ \mathbf{F} = \mathbf{I} + \nabla_0 \mathbf{u}, \quad \mathbf{C} = \mathbf{F}^T \mathbf{F}, \quad J = \det(\mathbf{F}) $$

 - $\mathbf{F}$: Deformation gradient
 - $\mathbf{C}$: Right Cauchy-Green deformation tensor
  - $J$: Determinant of deformation gradient

The strain energy density is defined as:

$$ \mathbf{W}(\mathbf{C}) = \frac{\mu}{2} \left(\text{tr}(\mathbf{C} - I) + \frac{2\mu}{\lambda}  J^{\frac{-\lambda}{2\mu}} -1\right) $$

The **2nd Piola-Kirchhoff stress tensor** is given by:

$$\mathbf{S} = 2 \frac{\partial \mathbf{W}}{\partial \mathbf{C}}$$

The **1st Piola-Kirchhoff stress tensor** is then computed as:

$$ \mathbf{P} = \mathbf{F}\mathbf{S} $$

#### **1.4 Boundary Conditions**

- **Dirichlet BC**: Fixed displacement on the fixed wall edges
- **Hinge Constraint**: average displacement on the `rotation` edge to zero in x- and y-direction
- **Neumann BC**: Normal traction on the `rotation` edge to apply external
- **Contact BC**: Contact condition between pendulum head edge and wall right edge using penalty method

------------------

### **2. Weak Formulation**

Multiplication of the governing PDE with a test function $\mathbf{v}$ and integration over the reference domain $\Omega_0$ yields the weak form:

$$
\int_{\Omega_0} \rho_0 \ddot{\mathbf{u}} \cdot \mathbf{v} \, dV + \int_{\Omega_0} \mathbf{P} : \nabla_0 \mathbf{v} \, dV = \int_{\Omega_0} \mathbf{b} \cdot \mathbf{v} \, dV + \int_{\Gamma_N} \mathbf{t}_0 \cdot \mathbf{v} \, dA
$$

 - $\Gamma_N$: Neumann boundary where traction $\mathbf{t}_0$ is applied
 - $\mathbf{v}$: Test function
 - $:$: Double contraction operator for tensors
- $dV$, $dA$: Volume and surface elements in the reference configuration
- $\nabla_0 \mathbf{v}$: Gradient of the test function with respect to the reference configuration

--------------------

### **3. Finite Element Spaces**

**Primary unknown: Displacement field $\mathbf{u}$**

$$V_h = [H^1(\Omega_0)]^2, \quad \mathbf{u}_h \in V_h$$

**Constraint variables for hinge condition:**

- Lagrange multipliers from a number space are used on the `rotation` edge to enforce the hinge constraint.

$$Q_h = \text{NumberSpace} \quad \text{on} \quad \Gamma_{rotation}$$

- Two number spaces are defined, one for each spatial direction (x and y).

- We obtain the mixed finite element space:

$$W_h = V_h \times Q_h \times Q_h = V_h \times Q_h^2$$

- This enforces the average displacement on the `rotation` edge to zero in both x- and y-direction and defined the center (0,0) of the edge as rotation point.

**Grid Functions:**

| **Symbol** | **Description** | **Code** |
|------------|-----------------|----------|
| $\mathbf{u}_h$ | Current Displacement Field | `self._gf_u` |
| $\mathbf{v}_h$ | Current Velocity Field | `self._gf_v` |
| $\mathbf{a}_h$ | Current Acceleration Field | `self._gf_a` |
| $\mathbf{u}_{old}$ | Displacement Field at previous time step | `self._gf_uold` |
| $\mathbf{v}_{old}$ | Velocity Field at previous time step | `self._gf_vold` |
| $\mathbf{a}_{old}$ | Acceleration Field at previous time step | `self._gf_aold` |

----------------------

### **4. Bilinear Form Setup**

Reference: https://docu.ngsolve.org/latest/i-tutorials/unit-3.4-nonlmin/nonlmin.html

The total weak form is build as a `BilinearForm` $a(\mathbf u, \mathbf v)$ in NGSolve by adding the different contributions:

**1. Strain Energy Contribution:**

Internal Strain Energy density:

$$ \mathbf{W}(\mathbf{C}) = \frac{\mu}{2} \left(\text{tr}(\mathbf{C} - I) + \frac{2\mu}{\lambda}  J^{\frac{-\lambda}{2\mu}} -1\right) $$

The variation gives the weak equilibrium: 

$$
\int_{\Omega} \mathbf{P} : \nabla \mathbf{v} \, d\Omega = 0
$$

where $\mathbf{P} = \frac{\partial W}{\partial \mathbf{F}}$ is the 1st Piola-Kirchhoff stress tensor.


In [ ]:
from ngsolve import Det, Grad, Id, InnerProduct, Trace, sqrt


class NeoHookeanMaterial:
    def __init__(self, E, nu):
        self.E = E
        self.nu = nu
        self.lmbda = (E * nu) / ((1 + nu) * (1 - 2 * nu))
        self.mu = E / (2 * (1 + nu))

    def C(self, u):
        F = Id(u.dim) + Grad(u)
        return F.trans * F

    def energy_density(self, C, u):
        return (
            0.5
            * self.mu
            * (
                Trace(C - Id(u.dim))
                + 2 * self.mu / self.lmbda * Det(C) ** (-self.lmbda / 2 / self.mu)
                - 1
            )
        )

    def sigma(self, C, u):
        return 2 * self.mu * (C - Id(u.dim)) + self.lmbda * Trace(C - Id(u.dim)) * Id(2)

In [ ]:
self = None  # to please the linter

# Bilinear form
self._bfa = BilinearForm(self._fes)
# Strain energy wall
if self._with_contact:
    self._bfa += Variation(
        self._material_law_w(self._deformation_gradient_w(self._u), self._u) * dx("wall")
    ).Compile()
    # Strain energy pendulum
    self._bfa += Variation(
        self._material_law_p(self._deformation_gradient_p(self._u), self._u) * dx("pendulum")
    ).Compile()

**2. Hinge Constraint Contribution:**

- Dirichlet boundary conditions for the wall are directly applied in the finite element space definition.
- Neumann boundary conditions for the hinge are weakly enforced through the variational formulation.

$$a(\mathbf u, \mathbf v) = a(\mathbf u, \mathbf v) + \int_{\Gamma_{rot}} (\mathbf u \cdot \mathbf p + \mathbf v \cdot \mathbf q) , ds$$


In [ ]:
# Rotation constraint
self._bfa += (InnerProduct(self._u, self._p) + InnerProduct(self._v, self._q)) * ds("rotation")

**3. Neumann Boundary Contribution (Drive Torque):**

- Drive torque is applied as traction on the `rotation` edge.

$$a(\mathbf u, \mathbf v) = a(\mathbf u, \mathbf v) + \int_{\Gamma_{rot}} (\mathbf{t} \cdot \mathbf v) , ds$$

In [ ]:
# Add traction (torque generating) to bfa as Neumann term on the rotation boundary
self._bfa += InnerProduct(self._t_drive, self._v) * ds("rotation")

**4. Inertial Contribution:**

Starting from $\int_{\Omega_0} \rho_0 \ddot{\mathbf{u}} \cdot \mathbf{v} \, dV$ we discretize in time using the Newmark-scheme and use the updates:

\begin{align}
\frac{u^{n+1}-u^n}{\tau} &= \frac{v^n+v^{n+1}}{2} \\
\frac{v^{n+1}-v^n}{\tau} &= \frac{a^n+a^{n+1}}{2}
\end{align}

\begin{align}
v^{n+1} &= \frac{2}{\tau}(u^{n+1} - u^n) - v^n \\
a^{n+1} &= \frac{2}{\tau}(v^{n+1} - v^n) - a^n
\end{align}

The inertial term becomes:

$$\int_{\Omega_0} \rho_0 a_{n+1} (\mathbf{u_{n+1}}) \cdot \mathbf{v} \, dV$$

In [ ]:
self.tau = Parameter(self.sim_params.tau)
vel_new = 2 / self.tau * (self._u - self._gf_uold.components[0]) - self._gf_vold.components[0]
acc_new = 2 / self.tau * (vel_new - self._gf_vold.components[0]) - self._gf_aold.components[0]

if self._with_contact:
    self._bfa += rhoA_w * InnerProduct(acc_new, self._v) * dx("wall")
self._bfa += rhoA_p * InnerProduct(acc_new, self._v) * dx("pendulum")

**5. Body Force Contribution (Gravity):**

- Gravity force is applied as body force in negative y-direction.

$$\int_{\Omega_0} \rho_0 \mathbf b \cdot \mathbf{v} \, dV \quad \text{with} \quad \rho_0 = \rho t$$


In [ ]:
#  gravity force
rhoA_p = self.rho_p * self.mat_params.thickness
rhoA_w = self.rho_w * self.mat_params.thickness
g = 9.81
if self._use_gravity:
    self._bfa += InnerProduct(CF((0, rhoA_w * g)), self._v) * dx("wall")
    self._bfa += InnerProduct(CF((0, rhoA_p * g)), self._v) * dx("pendulum")

**6. Contact Contribution (Penalty Method):**
- Defines the contact boundary between the contact edges of the ball and the wall
    - Master: contact edge of the wall
    - Slave: contact edge of the ball

- $X_M$: Master surface reference coordinates (undeformed configuration)

- $X_S$: Slave surface reference coordinates (undeformed configuration)

- $n_S$: Inverted normal vector of the slave surface (points outwards from the master/wall surface)

- Gap function `g`: scalar valued coefficient function that depends on the **normal gap** between master and slave surface

$$
\begin{align*}
    g_{\text{inc}} &= \left[(X_M + u_M - u_{old,M}) - (X_S + u_S - u_{old,S})\right] \cdot n_S
\end{align*}
$$

The gap function is then used in a **penalty energy**

$$\Pi_{\text{pen}} = \frac{1}{2} \int_{\Gamma_C} k_n \langle g_{\text{inc}}, 0 \rangle_+^2

In [ ]:
# Update contact with the current displacement
if self._with_contact:
    self._contact.Update(self._gf_u.components[0], self._bfa, intorder=10, maxdist=0.5)

`cb.Update(uold, bfmstar, intorder=10, maxdist=0.01, both_sides=False)`

- The contact is updated within each time step `contact.Update()` before solving the variational problem with `MinimizeNewton()`

- Displacement grid function `uold` is copied to an internal grid function

- The defined contact energy contributions are evaluated and inserted into the bilinear form `bfmstar`

- `intorder`: integration order for the master side quadrature and evaluating contact energy

- `maxdist`: maximum search distance (default: $2 \cdot \max{d_\text{element}}$)

-------------------

### **5. Torque Application**

**Idea:**
- Represent the desired torque as a distributed traction on the rotation edge $\Gamma_{rot}$
- Define the traction as:
$$\mathbf{t}_{drive}(s) = q_{drive} w_0(s) \mathbf{n}(s)$$

- where:
    - $q_{drive}$: scalar amplitude to scale the torque
    - $w_0(s)$: is a signed scalar weight along the edge
    - $\mathbf{n}(s)$: is the outward unit normal

**Idea:**

- The torque is converted into a **traction distribution** on the `rotation` edge of the pendulum rod.
- The net effect of this traction is to create the desired torque about the hinge point without creating any net force.
- The idea is to apply equal and opposite tractions on the hinge edge, such that the resultant force is zero, but the resultant moment (torque) is equal to the desired torque.
- A normal traction acting on the edge (pointing along the outward boundary normal).
- Traction are applied with opposite signs on opposite sides of the hinge point to create a pure moment

On a boundary $\Gamma_{rot}$ with outward normal vector $\mathbf{n}$, the 2D **resulting torque** from a boundary traction $\mathbf{t}$ is:

$$
M_z = \int_{\Gamma_{rot}} \left( \mathbf{r} \times \mathbf{t} \right) , ds
$$

 - $\mathbf{r} = (x,y)$: position relative to the hinge point
 - $\mathbf{t}$: traction vector on the boundary 

Setting the traction as:

$$
\mathbf{t}_{\text{drive}}(s) = q_{\text{drive}} w_0(s) \mathbf{n}(s)
$$

We obtain a normal traction, scaled by a signed weight function $w_0(s)$ along the edge and a scalar amplitude $q_{\text{drive}}$.

Then, the resulting torque becomes:

$$
M_z = q_{\text{drive}} \int_{\Gamma_{rot}} w_0(s) \left( \mathbf{r} \times \mathbf{n} \right), ds = q_{\text{drive}} D_{\text{pair}}
$$

- $q_{\text{drive}}$ is set as the desired torque divided by the torque distribution pair integral $D_{\text{pair}}$: $q_{\text{drive}} = \frac{M_{\text{desired}}}{D_{\text{pair}}}$

**Implementation Details**



In [ ]:
def _initialize_torque_control(self):
    # Motor torque
    self._normal_rot = specialcf.normal(2)
    r = self._X_rel

    # Smooth localized weight near rotation axis
    sigma = max(1e-9, 0.5 * self.geom_params.r_rod)
    w_core = exp(-(x * x) / (sigma * sigma))

    # Smooth splitter into two patches
    delta = 0.1 * sigma
    H = 0.5 * (1 + x / sqrt(x * x + delta * delta))  # smooth Heaviside function
    self._w_plus = w_core * H  # right patch weight
    self._w_minus = w_core * (1.0 - H)  # left patch weight

    self._cross_rn = -(r[0] * self._normal_rot[1] - r[1] * self._normal_rot[0])

    wdiff = self._w_plus - self._w_minus
    mean_w = Integrate(wdiff, self._mesh, definedon=self._mesh.Boundaries("rotation")) / Integrate(
        1, self._mesh, definedon=self._mesh.Boundaries("rotation")
    )
    wdiff0 = wdiff - mean_w
    self._q_drive = Parameter(0.0)
    self._t_drive = self._q_drive * wdiff0 * self._normal_rot
    self._effective_lever_arm = Integrate(
        wdiff0 * self._cross_rn, self._mesh, definedon=self._mesh.Boundaries("rotation")
    )

- `sigma`: width parameter to control the localization of the weight function
- `w_core`: Gaussian weight with localization around the hinge point
- `delta`: smoothing parameter for the Heaviside function
- `H`: Smooth Heaviside function to split the weight into two patches (left and right of hinge)
- `w_plus`, `w_minus`: Final weight functions for the two patches
- `w_diff`: Is roughly positive on the right and negative on the left of the hinge point
- `mean_w`: Average weight over the rotation boundary
- `wdiff0`: Zero-mean corrected weight difference (only net torque)
- `q_drive`: Controllable parameter to scale the applied torque
- `t_drive`: Traction vector applied on the rotation edge
- `D_pair`: Effectiveness factor
    - precomputes the integral of the weight function time the moment arm (`cross_rn`)
    - represents how much torque will be generated per unit of the input parameter `q_drive`
    - Allows to relate the desired torque to the traction amplitude

In [ ]:
def set_drive_torque(self, torque):
    Mz_2d = float(torque)  # (per unit thickness if 2D)
    q = Mz_2d / self._effective_lever_arm
    self._q_drive.Set(q)

- The desired torque $M_z$ is converted to a traction amplitude `q_drive` using the precomputed effectiveness factor `effective_lever_arm`
- This is then multiplied by the weight $w_0$ and the normal vector $\mathbf{n}$ and guarantees:

$$
\int_{\Gamma_{rot}} \left( \mathbf{r} \times \mathbf{t}_{drive} \right) ds = q_{drive} \int_{\Gamma_{rot}} w_0(s) \left( \mathbf{r} \times \mathbf{n} \right) ds = M_z
$$

-----------------------

## Background Information

#### **1. Modeling Elasticity**

##### **1.1. Kinematics**

|**Property**|**Definition**|
|---|---|
|Body in rest|$\Omega \subset \mathbb{R}^d$|
|Deformation function| $\phi : \Omega \rightarrow {\mathbb R}^3$|
|Displacement| $u(x) = \phi(x) - x$|
|Deformation Gradient| $F = \nabla \phi$|
|Cauchy-Green Strain Tensor| $C = F^T F$|
|Rigid Body Motion| $\phi(x) = a + Qx$, $a \in \mathbb{R}^3$, $Q$ is a rotation matrix |
|Green's Deformation Tensor| $E = \frac{1}{2}(C - I)=\frac{1}{2}\big( \nabla u + \nabla u^T + \nabla u^T \nabla u \big)$|


##### **1.2. Elastic Material Laws**

- When work is applied to a body, it deforms and stores deformation energy. 
- An elastic body returns the work when the external forces are removed.

**Hyperelastic materials**:

Constitutive law which expresses the deformation energy point-wise by the Cauchy-Green strain tensor:
$$
E_{def} = \int_\Omega W(C(u)) \, dx
$$

- Energy density function $W$ may depend on the position $x$ when the material of the body is inhomogeneous (i.e. $W = W(x, C(u))$).

**Isotropic constritution equation**:

- The material properties are the same in all direction.
- The deformation energy is independent of the rotation of the body before deformation.
- Thus, $W$ is a function the (real and positive) eigenvalues $\lambda_i$ of $E$
- **Characteristic polynomial**: $$\det (\lambda I - E) = \lambda^3 - I_1(E) \lambda^2 + I_2(E) \lambda - I_3(E)$$ with the invariants
\begin{align*}
        I_1(E) & =  \operatorname{tr} (E) = \lambda_1 + \lambda_2 + \lambda_3 \\
        I_2(E) & =  \frac{1}{2} [ (\operatorname{tr} E)^2 - \operatorname{tr} (E^2) ] = \lambda_1 \lambda_2 + \lambda_1 \lambda_3 + \lambda_2 \lambda_3 \\
        I_3(E) & =  \det (E) = \lambda_1 \lambda_2 \lambda_3
\end{align*}

- **Rivelin-Ericksen-Theorem** for isotropic materials: $$W(E) = W( \operatorname{tr} (E), \operatorname{tr}(E^2), \det (E) )$$
- Assuming that $W(E=0) = 0$ is a local minimum and $W$ is smooth, we can expand $W$ in a Taylor series around $E=0$:
$$
W(E) = \frac{1}{2} W_{,11} \operatorname{tr} (E)^2 + W_{,2} \operatorname{tr} (E^2) + O(\| E \|^3)
$$

- Dropping higher order terms and Lamé parameters $\lambda := W_{,11}$ (resistance to volumetric deformation) and $\mu := W_{,2}$ (resistance to shear deformation) we obtain the second order energy density (called *Hooke's Law*):
$$
W(E) = \frac{\lambda}{2} \operatorname{tr} (E)^2 + \mu E:E,
$$ 
- It is similar to an elastic spring: The stored energy is $\frac{1}{2} k E^2$, with the spring constant $k$ and elongation $E$.

##### **1.3. Variational Formulation and Equilibrium**

- Let $V$ be a function space of feasible displacements and $\Gamma_D$ the Dirichlet boundary where the displacement is prescribed

- **Total energy = Deformation energy + Potential of external forces**:
$$
J(u) = \int_\Omega W(C(u)) \, dx - \int_\Omega f \cdot u \, dx + \text{Inertia(u)} + \text{Torque(u)}
$$
with 
  - $f$: volume force density
  - Inertia (as a function of displacement $u$, Newmark scheme for $a, v$ updates)
  - Torque is applied as a distributed body force

#### **2. Solving Nonlinear Elasticity Variational Problems**

We consider minimization problems of the form

$$\text{find } u \in V \text{ s.t. } E(u) \leq E(v) \quad \forall~  v \in V.$$

- We are solving this problem using Newton's method.
- We are using the `Variation` integrator of `NGSolve` and formulate the problem through a symbolic description of an energy functional.
- Let $E(u)$ be the energy that is to be minimized for the unknown state $u$.
- A necessary optimality condition is that the derivative at the minimizer $u$ in all directions $v$ vanishes, i.e. 
$$
  \delta E(u) (v) = 0 \quad \forall v \in V
$$

- We assume for our pendulum a Neo-Hookean hyperelastic material model.
- The energy density function is given as
$$
  E(v) := \int_{\Omega} \frac{\mu}{2} ( \operatorname{tr}(F^T F-I)+\frac{2 \mu}{\lambda} \operatorname{det}(F^T F)^{-\frac{\lambda}{2\mu}} - 1) ~~ dx
$$

#### **3. Solving Dynamic Contact Problems**

- Dynamic contact combines for our example nonlinear elasticity with contact constraints in a time-stepping scheme.
- The approach uses a penalty method to enforce the contact conditions and to handle large deformations through the Neo-Hookean material models.

1. **Geometry and Mesh**:
- Contact boundaries must be explicitly defined
- Self-contact is supported (same boundary can contact itself)

2. **Contact Gap Function**:

- Measures the signed distance between the two contact surface
- Negative values indicate penetration

``` python
cf = (X + u-uold - (X.Other() + u.Other() - uold.Other())) * (-specialcf.normal(2).Other())
```
Where 
- `X`: reference position
- `u`: current displacement
- `uold`: displacement from previous time step
- `Other()`: values from the other side of the contact boundary
- `specialcf.normal(2)`: normal vector on the contact boundary (pointing outward from the element)
- `cf`: gap function, negative values indicate penetration

3. **Contact Energy**:

``` python
contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=True)
```
- Penalty method: large stiffness ($1e9$) when surfaces penetrate (cf < 0)
- `IfPos(cf, 1e9*cf*cf, 0)`: adds energy only when `cf` is positive (penetration)
- `deformed=True`: evaluates the energy in the deformed configuration

#### **4. Newmark time-stepping**

References:
- [Newmark-beta method - Wikipedia](https://en.wikipedia.org/wiki/Newmark-beta_method)
- [NGSolve Documentation - Newmark Method](https://docu.ngsolve.org/ngs24/SaS/dynamics_newmark_gen_alpha.html)

The Newmark method is an implicit time-integration scheme for solving second-order differential equations (structural dynamics problems).

1. **Mathematical Foundation**
- The scheme solves the dynamic equilibrium equations by approximating the displacement, velocity, and acceleration at each time step.
$$
M \ddot{u} + C \dot{u} + f^{int} u = f^{ext}
$$
- where:
    - $M$: mass matrix
    - $C$: damping matrix 
    - $f^{int}$: internal force vector
    - $f^{ext}$: external force vector

- New acceleration is obtained from the elasticity operator $K$:
$$
a^{n+1} = f - K(u^{n+1})
$$
- where:
    - Displacement $u^{n+1}$ and the acceleration $a^{n+1}$ at the new time step are unknowns
    - The velocity has to be determined via the time stepping scheme (see below)

2. **Newmark Scheme**

- Trapezoidal rule (average acceleration method) is used to approximate the velocity and acceleration.
- Implicit method: requires solving a nonlinear system at each time step (e.g., using Newton's method).

Key equations of the Newmark method are:

\begin{align}
\frac{u^{n+1}-u^n}{\tau} &= \frac{v^n+v^{n+1}}{2} \\
\frac{v^{n+1}-v^n}{\tau} &= \frac{a^n+a^{n+1}}{2}
\end{align}

They can be rearranged to express $v^{n+1}$ and $a^{n+1}$ in terms of $u^{n+1}$:

\begin{align}
v^{n+1} &= \frac{2}{\tau}(u^{n+1} - u^n) - v^n \\
a^{n+1} &= \frac{2}{\tau}(v^{n+1} - v^n) - a^n
\end{align}

3. **Procedure**

    1. **Prediction:** Start with known values at time step $n$ ($u^n$, $v^n$, $a^n$)
    2. **Implicit Solution:** Solve nonlinear system for $u^{n+1}$ using Newton's method.
    3. **Correction:** Update $v^{n+1}$ and $a^{n+1}$ using the equations above.
    4. **Advance:** Move to the next time step and repeat.

#### **5. Constraint Implementation - Pendulum Pivot**

```python
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('top')
```

- Lagrange multipliers `p` and `q` are introduced to enforce the constraints at the pendulum pivot.
- Mean value constraint: Controls the average displacement of pivot point.
- Allows the pendulum to swing freely while keeping the pivot fixed.

#### **6. Variational Form - Weak Formulation**

- **Principle of Virtual Work**: The weak form of the equilibrium equations is derived from the principle of virtual work, which states that the work done by internal forces equals the work done by external forces for any virtual displacement
$$
\delta W_{\text{internal}} = \delta W_{\text{external}}
$$

- `Variation(NeoHooke(C(u))*dx).Compile()`: Computes the variation of the Neo-Hookean energy density integrated over the domain. Corresponds to the internal elastic work from material nonlinearity.
- `acc_new*v*dx`: Represents the inertial forces due to acceleration.
- `-force*v*dx`: Represents the work done by external forces (e.g., gravity).


#### **7. Key Points**

1. **Large Deformations**: \
The Neo-Hookean material model captures large deformations and nonlinear elasticity.

2. **Contact Handling**: \
The penalty method effectively enforces contact constraints, preventing interpenetration.

3. **Time Integration**: \
The Newmark method provides a stable and accurate time-stepping scheme for dynamic simulations.

4. **Constraint Enforcement**: \
Lagrange multipliers are used to enforce constraints at the pendulum pivot, allowing for realistic motion.

5. **Variational Formulation**: \
The weak form derived from the principle of virtual work ensures that the internal and external forces are balanced for any virtual displacement.

--------------------------------------